# KrishiRakshak — AI-Powered Crop Disease Diagnosis

| Notebook Section | Production File |
|---|---|
| 0. Setup | — |
| 1. Dataset Preparation | `training/prepare_dataset.py` |
| 2. Florence-2 Fine-Tuning | `training/fine_tune_florence.py` |
| 3. Model Evaluation | `training/evaluate_model.py` |
| 4. Push to SageMaker | `training/push_to_sagemaker.py` |
| 5. Image Preprocessor | `src/pipeline/image_preprocessor.py` |
| 6. Inference Service | `src/models/florence_inference.py` |
| 7. S3 Service | `src/services/s3_service.py` |
| 8. Feedback Service | `src/services/feedback_service.py` |
| 9. Translation Service | `src/services/sarvam_translate.py` |
| 10. TTS Service | `src/services/sarvam_tts.py` |
| 11. Full Pipeline | `src/pipeline/diagnosis_pipeline.py` |
| 12. API Layer | `src/api/` |
| 13. Monitoring | `src/monitoring/metrics.py` |

---
## 0. Setup

In [ ]:
!pip install -q transformers peft torch torchvision pillow scikit-learn boto3 sagemaker pyyaml httpx tqdm structlog fastapi uvicorn pydantic
print('✓ Dependencies installed')

In [ ]:
import os, json, logging, random, time, uuid, base64, io, re, shutil, tarfile
from pathlib import Path
from dataclasses import dataclass
from collections import Counter
from datetime import datetime, timezone
import boto3, yaml, torch, httpx
from PIL import Image, ExifTags
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

PROJECT_ROOT = Path('/home/ec2-user/SageMaker/krishirakshak-project')
os.chdir(PROJECT_ROOT)
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger(__name__)
S3_BUCKET = 'krishirakshak-data-593755927741'

print(f'✓ Working directory: {os.getcwd()}')
print(f'✓ PyTorch version: {torch.__version__}')
print(f'✓ CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'✓ S3 Bucket: {S3_BUCKET}')

# Verify project files exist
required_files = ['training/training_config.yaml','training/treatment_kb.json','configs/app_config.yaml','configs/sarvam_config.yaml','configs/model_config.yaml']
for f in required_files:
    status = '✓' if Path(f).exists() else '✗ MISSING'
    print(f'  {status} {f}')

---
## 1. Dataset Preparation
**Production file:** `training/prepare_dataset.py`

Converts raw PlantVillage images into Florence-2 training format:
- Input: `<CROP_DISEASE>` (prompt)
- Target: `Disease Name. Severity: X. Symptoms: Y. Immediate Action: Z. Pesticide: P...`

In [ ]:
# Load and display training config
with open('training/training_config.yaml') as f:
    config = yaml.safe_load(f)
print('=== Training Configuration ===')
print(f'  Model:                {config["model"]["name"]}')
print(f'  LoRA rank:            {config["lora"]["r"]}')
print(f'  LoRA alpha:           {config["lora"]["lora_alpha"]}')
print(f'  LoRA target modules:  {config["lora"]["target_modules"]}')
print(f'  Epochs:               {config["training"]["epochs"]}')
print(f'  Batch size:           {config["training"]["batch_size"]}')
print(f'  Gradient accumulation:{config["training"]["gradient_accumulation_steps"]}')
print(f'  Effective batch size: {config["training"]["batch_size"] * config["training"]["gradient_accumulation_steps"]}')
print(f'  Learning rate:        {config["training"]["learning_rate"]}')
print(f'  FP16:                 {config["training"]["fp16"]}')
print(f'  Freeze vision encoder:{config["training"]["freeze_vision_encoder"]}')
print(f'  Train/Val/Test split: {config["dataset"]["train_split"]}/{config["dataset"]["val_split"]}/{config["dataset"]["test_split"]}')
print(f'  Min accuracy threshold:{config["evaluation"]["min_accuracy_threshold"]}')
print('✓ Config loaded')

In [ ]:
# Sync dataset from S3
RAW_DATA_DIR = Path('/tmp/plantvillage')
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f'Syncing from s3://{S3_BUCKET}/data/raw/plantvillage ...')
ret = os.system(f'aws s3 sync s3://{S3_BUCKET}/data/raw/plantvillage {RAW_DATA_DIR} --quiet')
print(f'  Exit code: {ret} ({"success" if ret==0 else "FAILED"})')

# Verify sync
all_images = list(RAW_DATA_DIR.rglob('*.jpg')) + list(RAW_DATA_DIR.rglob('*.JPG')) + list(RAW_DATA_DIR.rglob('*.png'))
print(f'  Total images found: {len(all_images)}')
assert len(all_images) > 0, 'ERROR: No images found after S3 sync!'
print('✓ S3 sync complete')

In [ ]:
# Load treatment knowledge base
with open('training/treatment_kb.json') as f:
    treatment_kb = json.load(f)['diseases']
print(f'✓ Loaded treatment KB: {len(treatment_kb)} diseases')

# Verify all expected disease classes are present
expected_classes = ['Tomato Early Blight','Potato Late Blight','Apple Scab','Grape Black Rot']
for cls in expected_classes:
    status = '✓' if cls in treatment_kb else '✗ MISSING'
    print(f'  {status} {cls}')

# Show sample entry
print('\nSample KB entry — Tomato Early Blight:')
entry = treatment_kb.get('Tomato Early Blight', {})
print(f'  Severity: {entry.get("severity")}')
print(f'  Symptoms: {entry.get("symptoms")}')
t = entry.get('treatment', {})
print(f'  Immediate action: {t.get("immediate_action")}')
print(f'  Pesticide: {t.get("pesticide")}')
print(f'  Application: {t.get("application")}')

In [ ]:
# map_folder_to_disease() — maps PlantVillage folder names to KB disease names
def map_folder_to_disease(folder_name):
    mapping = {
        'Apple___Apple_scab':'Apple Scab','Apple___Black_rot':'Apple Black Rot',
        'Apple___Cedar_apple_rust':'Apple Cedar Rust','Apple___healthy':'Apple Healthy',
        'Blueberry___healthy':'Blueberry Healthy',
        'Cherry_(including_sour)___Powdery_mildew':'Cherry Powdery Mildew',
        'Cherry_(including_sour)___healthy':'Cherry Healthy',
        'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot':'Corn Cercospora Leaf Spot',
        'Corn_(maize)___Common_rust_':'Corn Common Rust',
        'Corn_(maize)___Northern_Leaf_Blight':'Corn Northern Leaf Blight',
        'Corn_(maize)___healthy':'Corn Healthy','Grape___Black_rot':'Grape Black Rot',
        'Grape___Esca_(Black_Measles)':'Grape Esca (Black Measles)',
        'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)':'Grape Leaf Blight',
        'Grape___healthy':'Grape Healthy',
        'Orange___Haunglongbing_(Citrus_greening)':'Orange Citrus Greening',
        'Peach___Bacterial_spot':'Peach Bacterial Spot','Peach___healthy':'Peach Healthy',
        'Pepper,_bell___Bacterial_spot':'Bell Pepper Bacterial Spot',
        'Pepper,_bell___healthy':'Bell Pepper Healthy',
        'Potato___Early_blight':'Potato Early Blight','Potato___Late_blight':'Potato Late Blight',
        'Potato___healthy':'Potato Healthy','Raspberry___healthy':'Raspberry Healthy',
        'Soybean___healthy':'Soybean Healthy','Squash___Powdery_mildew':'Squash Powdery Mildew',
        'Strawberry___Leaf_scorch':'Strawberry Leaf Scorch','Strawberry___healthy':'Strawberry Healthy',
        'Tomato___Bacterial_spot':'Tomato Bacterial Spot','Tomato___Early_blight':'Tomato Early Blight',
        'Tomato___Late_blight':'Tomato Late Blight','Tomato___Leaf_Mold':'Tomato Leaf Mold',
        'Tomato___Septoria_leaf_spot':'Tomato Septoria Leaf Spot',
        'Tomato___Spider_mites Two-spotted_spider_mite':'Tomato Spider Mites',
        'Tomato___Target_Spot':'Tomato Target Spot',
        'Tomato___Tomato_Yellow_Leaf_Curl_Virus':'Tomato Yellow Leaf Curl Virus',
        'Tomato___Tomato_mosaic_virus':'Tomato Mosaic Virus','Tomato___healthy':'Tomato Healthy',
    }
    return mapping.get(folder_name, folder_name)

# Test mapping
test_cases = [('Tomato___Early_blight','Tomato Early Blight'),('Apple___Apple_scab','Apple Scab')]
for folder, expected in test_cases:
    result = map_folder_to_disease(folder)
    status = '✓' if result == expected else f'✗ GOT {result}'
    print(f'  {status} {folder} → {expected}')
print(f'\n✓ Folder mapping defined ({len(test_cases)} spot-checks passed)')

In [ ]:
# format_treatment_text() — builds the training target string
def format_treatment_text(disease_name, treatment_kb):
    if disease_name not in treatment_kb:
        return f'{disease_name}. Treatment information not available.'
    info = treatment_kb[disease_name]
    t = info['treatment']
    return (f'{disease_name}. Severity: {info["severity"]}. Symptoms: {info["symptoms"]}. '
            f'Immediate Action: {t["immediate_action"]} Pesticide: {t["pesticide"]}. '
            f'Application: {t["application"]}. Prevention: {t["prevention"]}')

# Verify output format
sample = format_treatment_text('Tomato Early Blight', treatment_kb)
print('Sample treatment text (Florence-2 training target):')
print(f'  {sample}')
print(f'\n  Length: {len(sample)} chars')
assert 'Tomato Early Blight' in sample
assert 'Severity' in sample
assert 'Pesticide' in sample
print('✓ format_treatment_text() checks passed')

In [ ]:
# build_dataset() — scan PlantVillage folders, create one entry per image
def build_dataset(raw_dir, treatment_kb):
    entries = []
    search_dirs = [raw_dir]
    for sub in raw_dir.iterdir():
        if sub.is_dir():
            search_dirs.append(sub)
            for subsub in sub.iterdir():
                if subsub.is_dir(): search_dirs.append(subsub)

    image_root = None
    for d in search_dirs:
        subdirs = [x for x in d.iterdir() if x.is_dir()]
        if len(subdirs) >= 30:
            image_root = d
            break

    if image_root is None:
        raise FileNotFoundError(f'Could not find PlantVillage class folders in {raw_dir}')

    print(f'  Image root found: {image_root}')
    class_dirs = [d for d in sorted(image_root.iterdir()) if d.is_dir()]
    print(f'  Total class folders in PlantVillage: {len(class_dirs)}')

    for class_dir in class_dirs:
        disease_name = map_folder_to_disease(class_dir.name)
        treatment_text = format_treatment_text(disease_name, treatment_kb)
        images = list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.JPG')) + list(class_dir.glob('*.png'))
        for img in images:
            entries.append({'image_path':str(img),'prefix':'<CROP_DISEASE>','suffix':treatment_text,'disease_name':disease_name,'class_folder':class_dir.name})

    return entries

# --- Top 5 diseases for fine-tuning demo ---
# Chosen for: crop importance in India + visually distinct symptoms
# Full 38-class training would need more GPU time and data — same code, just remove filter
TOP_5_DISEASES = [
    'Tomato Early Blight',   # Very common, brown lesions with concentric rings
    'Tomato Late Blight',    # Severe, causes total crop loss
    'Potato Late Blight',    # Same pathogen as Tomato Late Blight (Phytophthora infestans)
    'Tomato Leaf Mold',      # Yellow patches on upper leaf surface
    'Corn Common Rust',      # Brick-red pustules on corn leaves
]
print('Target diseases for fine-tuning:')
for i, d in enumerate(TOP_5_DISEASES):
    print(f'  {i+1}. {d}')

print('\nBuilding full dataset from PlantVillage...')
all_entries = build_dataset(RAW_DATA_DIR, treatment_kb)
print(f'\n✓ Full dataset (38 classes): {len(all_entries)} entries')

# Filter to top 5
entries = [e for e in all_entries if e['disease_name'] in TOP_5_DISEASES]
print(f'✓ Filtered to top 5 diseases: {len(entries)} entries')
assert len(entries) > 0, 'No entries found for top 5 diseases!'

print('\nClass distribution (filtered dataset):')
for disease in TOP_5_DISEASES:
    count = sum(1 for e in entries if e['disease_name'] == disease)
    bar = '█' * (count // 100)
    print(f'  {disease:<30} {count:>5} images  {bar}')

print(f'\n✓ Sample entry keys: {list(entries[0].keys())}')
print(f'  prefix: {entries[0]["prefix"]}')
print(f'  suffix (first 100 chars): {entries[0]["suffix"][:100]}')

In [ ]:
# split_and_save() — stratified train/val/test split, save as JSONL
PROCESSED_DIR = Path('data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

labels = [e['disease_name'] for e in entries]
unique_classes = sorted(set(labels))
print(f'Unique classes: {len(unique_classes)}')

# Check class distribution
class_counts = Counter(labels)
min_class = min(class_counts, key=class_counts.get)
max_class = max(class_counts, key=class_counts.get)
print(f'  Largest class:  {max_class} ({class_counts[max_class]} images)')
print(f'  Smallest class: {min_class} ({class_counts[min_class]} images)')
print(f'  Imbalance ratio: {class_counts[max_class]/class_counts[min_class]:.1f}x')

train_ratio = config['dataset']['train_split']
val_ratio   = config['dataset']['val_split']

train_entries, temp, _, temp_labels = train_test_split(entries, labels, test_size=(1-train_ratio), stratify=labels, random_state=42)
val_entries, test_entries = train_test_split(temp, test_size=0.5, stratify=temp_labels, random_state=42)

for split_name, split_data in [('train',train_entries),('val',val_entries),('test',test_entries)]:
    out = PROCESSED_DIR / f'{split_name}.jsonl'
    with open(out,'w') as f:
        for e in split_data: f.write(json.dumps(e)+'\n')
    # Verify file written
    lines = sum(1 for _ in open(out))
    print(f'  ✓ {split_name}.jsonl: {lines} entries ({out.stat().st_size/1024:.1f} KB)')

with open(PROCESSED_DIR/'class_map.json','w') as f:
    json.dump({i:name for i,name in enumerate(unique_classes)},f,indent=2)
print(f'  ✓ class_map.json: {len(unique_classes)} classes')

# Upload to S3
print('\nUploading processed data to S3...')
ret = os.system(f'aws s3 sync data/processed s3://{S3_BUCKET}/data/processed --quiet')
print(f'✓ Upload complete (exit code: {ret})')

---
## 2. Florence-2 Fine-Tuning with LoRA
**Production file:** `training/fine_tune_florence.py`

**Why freeze the vision encoder?** Florence-2's image encoder is already pretrained on billions of images — it knows how to see. We only need to teach the decoder to generate crop disease text. Freezing saves GPU memory and training time.

**Why LoRA rank=8?** Rank controls adapter capacity. r=8 gives ~2M trainable params vs 232M total — enough to specialize for 38 disease classes without overfitting.

In [ ]:
from transformers import AutoProcessor, AutoModelForCausalLM, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, PeftModel
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

# Data augmentation transforms — applied only during TRAINING, not validation/test
# Mirrors augmentations listed in training_config.yaml
TRAIN_AUGMENTATIONS = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=15),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    T.RandomResizedCrop(size=224, scale=(0.8, 1.0)),
])

print('Data augmentation pipeline:')
print('  RandomHorizontalFlip(p=0.5)         — mirrors leaf image left/right')
print('  RandomRotation(15°)                 — handles different leaf angles')
print('  ColorJitter(brightness, contrast)   — handles different lighting conditions')
print('  RandomResizedCrop(224, scale=0.8-1) — handles different zoom levels')
print()
print('Why augmentation matters for 5 classes:')
print('  With only 5 diseases, model can overfit easily.')
print('  Augmentation artificially expands the dataset and improves generalization.')


class CropDiseaseDataset(Dataset):
    """
    Florence-2 compatible dataset with optional data augmentation.
    Mirrors CropDiseaseDataset in fine_tune_florence.py.
    augment=True for training, augment=False for val/test.
    """
    def __init__(self, jsonl_path, processor, max_length=512, augment=False):
        self.processor = processor
        self.max_length = max_length
        self.augment = augment
        self.entries = []
        with open(jsonl_path) as f:
            for line in f: self.entries.append(json.loads(line.strip()))
        print(f'  Loaded {len(self.entries)} entries from {jsonl_path} (augment={augment})')

    def __len__(self): return len(self.entries)

    def __getitem__(self, idx):
        e = self.entries[idx]
        image = Image.open(e['image_path']).convert('RGB')

        # Apply augmentation only on training set
        if self.augment:
            image = TRAIN_AUGMENTATIONS(image)

        inputs = self.processor(text=e['prefix'], images=image, return_tensors='pt',
                                padding='max_length', max_length=self.max_length, truncation=True)
        labels = self.processor.tokenizer(e['suffix'], return_tensors='pt',
                                          padding='max_length', max_length=self.max_length, truncation=True)

        input_ids    = inputs['input_ids'].squeeze(0)
        pixel_values = inputs['pixel_values'].squeeze(0)
        label_ids    = labels['input_ids'].squeeze(0)
        label_ids[label_ids == self.processor.tokenizer.pad_token_id] = -100  # ignore padding in loss
        return {'input_ids': input_ids, 'pixel_values': pixel_values, 'labels': label_ids}


def collate_fn(batch):
    return {'input_ids': torch.stack([b['input_ids'] for b in batch]),
            'pixel_values': torch.stack([b['pixel_values'] for b in batch]),
            'labels': torch.stack([b['labels'] for b in batch])}

print('\n✓ CropDiseaseDataset with augmentation defined')

In [ ]:
# Load Florence-2 + apply LoRA
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

print(f'Loading {config["model"]["name"]} ...')
processor = AutoProcessor.from_pretrained(config['model']['name'], trust_remote_code=True, revision=config['model']['revision'])
model = AutoModelForCausalLM.from_pretrained(config['model']['name'], trust_remote_code=True, revision=config['model']['revision'], torch_dtype=torch.float16 if config['training']['fp16'] else torch.float32)
print(f'✓ Model loaded. Total params: {sum(p.numel() for p in model.parameters()):,}')

# Freeze vision encoder
if config['training']['freeze_vision_encoder']:
    frozen = 0
    for param in model.vision_tower.parameters():
        param.requires_grad = False
        frozen += param.numel()
    print(f'✓ Vision encoder frozen: {frozen:,} params frozen')

# Apply LoRA
lc = config['lora']
lora_config = LoraConfig(r=lc['r'],lora_alpha=lc['lora_alpha'],lora_dropout=lc['lora_dropout'],target_modules=lc['target_modules'],bias=lc['bias'],task_type=lc['task_type'])
model = get_peft_model(model, lora_config)
model.to(DEVICE)
print('\n✓ LoRA applied. Trainable parameter breakdown:')
model.print_trainable_parameters()

In [ ]:
# Load datasets — augment=True for train only, False for val/test
print('Loading datasets...')
train_dataset = CropDiseaseDataset('data/processed/train.jsonl', processor, config['training']['max_length'], augment=True)
val_dataset   = CropDiseaseDataset('data/processed/val.jsonl',   processor, config['training']['max_length'], augment=False)

# Verify a sample batch shape
sample = train_dataset[0]
print(f'\n✓ Sample batch shapes:')
print(f'  input_ids:    {sample["input_ids"].shape}')
print(f'  pixel_values: {sample["pixel_values"].shape}')
print(f'  labels:       {sample["labels"].shape}')
print(f'  Labels with -100 (padding ignored): {(sample["labels"]==-100).sum().item()}')

# Visualize augmentation effect
import matplotlib.pyplot as plt
orig_img  = Image.open(train_dataset.entries[0]['image_path']).convert('RGB')
aug_img   = TRAIN_AUGMENTATIONS(orig_img)
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(orig_img.resize((224,224))); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(aug_img);                   axes[1].set_title('Augmented'); axes[1].axis('off')
plt.suptitle('Data Augmentation Effect')
plt.tight_layout(); plt.show()
print('✓ Augmentation visualization above — each training epoch sees differently transformed images')

train_loader = DataLoader(train_dataset, batch_size=config['training']['batch_size'], shuffle=True,  collate_fn=collate_fn, num_workers=config['training']['dataloader_num_workers'], pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=config['training']['batch_size'], shuffle=False, collate_fn=collate_fn, num_workers=config['training']['dataloader_num_workers'])
print(f'\n✓ DataLoaders ready')
print(f'  Train batches: {len(train_loader)} (augmented)')
print(f'  Val batches:   {len(val_loader)} (no augmentation)')

In [ ]:
# Training loop — mirrors train() in fine_tune_florence.py
optimizer = torch.optim.AdamW(model.parameters(), lr=config['training']['learning_rate'], weight_decay=config['training']['weight_decay'])
num_training_steps = len(train_loader) * config['training']['epochs']
warmup_steps = int(num_training_steps * config['training']['warmup_ratio'])
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=num_training_steps)

output_dir    = Path(config['paths']['output_dir'])
best_model_dir= Path(config['paths']['best_model'])
output_dir.mkdir(parents=True, exist_ok=True)
best_model_dir.mkdir(parents=True, exist_ok=True)

grad_accum = config['training']['gradient_accumulation_steps']
best_val_loss = float('inf')

print(f'Starting training for {config["training"]["epochs"]} epochs')
print(f'  Warmup steps: {warmup_steps} / {num_training_steps} total')
print(f'  Output dir: {output_dir}')
print(f'  Best model dir: {best_model_dir}')

for epoch in range(config['training']['epochs']):
    model.train()
    epoch_loss, num_batches = 0.0, 0

    for step, batch in enumerate(train_loader):
        batch = {k: v.to(DEVICE) for k,v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss / grad_accum
        loss.backward()

        if (step+1) % grad_accum == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step(); optimizer.zero_grad()

        epoch_loss += outputs.loss.item(); num_batches += 1

        if (step+1) % config['training']['logging_steps'] == 0:
            print(f'  Epoch {epoch+1} | Step {step+1}/{len(train_loader)} | Loss: {epoch_loss/num_batches:.4f} | LR: {scheduler.get_last_lr()[0]:.2e}')

    # Validation
    model.eval(); val_loss, val_batches = 0.0, 0
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(DEVICE) for k,v in batch.items()}
            val_loss += model(**batch).loss.item(); val_batches += 1

    avg_train = epoch_loss/num_batches
    avg_val   = val_loss/val_batches
    print(f'\nEpoch {epoch+1}/{config["training"]["epochs"]} COMPLETE')
    print(f'  Train Loss: {avg_train:.4f}')
    print(f'  Val Loss:   {avg_val:.4f}')

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        model.save_pretrained(best_model_dir)
        processor.save_pretrained(best_model_dir)
        saved_files = list(best_model_dir.iterdir())
        print(f'  ✓ Best model saved to {best_model_dir} ({len(saved_files)} files)')
    else:
        print(f'  (No improvement. Best val loss still: {best_val_loss:.4f})')

    if (epoch+1) % 2 == 0:
        ckpt = output_dir/f'checkpoint-epoch{epoch+1}'
        ckpt.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(ckpt)
        print(f'  ✓ Checkpoint saved: {ckpt}')

print(f'\n✓ Training complete! Best val loss: {best_val_loss:.4f}')
print(f'✓ Best model at: {best_model_dir}')

---
## 3. Model Evaluation
**Production file:** `training/evaluate_model.py`

Runs inference on held-out test set and computes accuracy/F1. A **pass/fail gate** at 90% accuracy prevents bad models from being deployed.

In [ ]:
# Load fine-tuned model for evaluation
def load_finetuned_model(model_path, device):
    print(f'Loading fine-tuned model from {model_path} ...')
    assert Path(model_path).exists(), f'Model path not found: {model_path}'
    files = list(Path(model_path).iterdir())
    print(f'  Model files: {[f.name for f in files]}')
    proc = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)
    base = AutoModelForCausalLM.from_pretrained('microsoft/Florence-2-base-ft', trust_remote_code=True, torch_dtype=torch.float16)
    m = PeftModel.from_pretrained(base, model_path)
    m.to(device).eval()
    print(f'✓ Model loaded on {device}')
    return m, proc

def extract_disease_name(output_text):
    """Output format: 'Disease Name. Severity: ...' — take first sentence."""
    parts = output_text.split('.')
    return parts[0].strip() if parts else 'Unknown'

eval_model, eval_processor = load_finetuned_model(config['paths']['best_model'], DEVICE)

In [ ]:
# Run evaluation on test set
test_entries = []
with open('data/processed/test.jsonl') as f:
    for line in f: test_entries.append(json.loads(line.strip()))
print(f'Test samples: {len(test_entries)}')

true_labels, pred_labels, results = [], [], []
errors_count = 0

for i, entry in enumerate(test_entries):
    try:
        image = Image.open(entry['image_path']).convert('RGB')
        inputs = eval_processor(text='<CROP_DISEASE>', images=image, return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            out = eval_model.generate(**inputs, max_new_tokens=512, num_beams=3)
        raw = eval_processor.batch_decode(out, skip_special_tokens=True)[0]
        pred = extract_disease_name(raw)
        true_labels.append(entry['disease_name'])
        pred_labels.append(pred)
        correct = entry['disease_name'].lower() == pred.lower()
        results.append({'true':entry['disease_name'],'predicted':pred,'correct':correct,'raw':raw[:150]})
        if (i+1) % 100 == 0:
            running_acc = sum(1 for r in results if r['correct'])/len(results)
            print(f'  Progress: {i+1}/{len(test_entries)} | Running accuracy: {running_acc:.1%}')
    except Exception as e:
        errors_count += 1
        logger.warning(f'Failed on {entry["image_path"]}: {e}')
        true_labels.append(entry['disease_name']); pred_labels.append('ERROR')

print(f'\nEvaluation complete. Errors/skipped: {errors_count}')

accuracy = accuracy_score(true_labels, pred_labels)
unique_labels = sorted(set(true_labels))
report = classification_report(true_labels, pred_labels, labels=unique_labels, output_dict=True, zero_division=0)

print(f'\n=== EVALUATION RESULTS ===')
print(f'  Total samples:  {len(test_entries)}')
print(f'  Accuracy:       {accuracy:.4f} ({accuracy*100:.1f}%)')
print(f'  Weighted F1:    {report["weighted avg"]["f1-score"]:.4f}')
print(f'  Macro F1:       {report["macro avg"]["f1-score"]:.4f}')

class_f1 = {k:v['f1-score'] for k,v in report.items() if k in unique_labels}
worst = sorted(class_f1.items(), key=lambda x:x[1])[:5]
print(f'\n  Worst 5 classes (need more training data or augmentation):')
for cls,f1 in worst: print(f'    {cls}: F1={f1:.3f}')

error_pairs = Counter((r['true'],r['predicted']) for r in results if not r['correct'])
print(f'\n  Top 5 confusion pairs:')
for (t,p),cnt in error_pairs.most_common(5): print(f'    {t} → {p}: {cnt} times')

# Save results
Path('outputs/evaluation').mkdir(parents=True, exist_ok=True)
with open('outputs/evaluation/metrics_summary.json','w') as f:
    json.dump({'accuracy':accuracy,'weighted_f1':report['weighted avg']['f1-score'],'macro_f1':report['macro avg']['f1-score'],'total_samples':len(test_entries)},f,indent=2)
print(f'\n✓ Results saved to outputs/evaluation/')

# Pass/fail gate — mirrors evaluate_model.py
min_acc = config['evaluation']['min_accuracy_threshold']
if accuracy >= min_acc:
    print(f'\n✓ PASSED: {accuracy:.3f} >= threshold {min_acc}. Safe to deploy.')
else:
    print(f'\n✗ FAILED: {accuracy:.3f} < threshold {min_acc}. DO NOT deploy — retrain needed.')

---
## 4. Package and Deploy to SageMaker
**Production file:** `training/push_to_sagemaker.py`

Packages LoRA adapter weights + inference handler into `model.tar.gz` and deploys as a **Serverless Endpoint** (zero cost when idle).

In [ ]:
# package_model() — mirrors push_to_sagemaker.py
def package_model(config):
    model_dir  = Path(config['paths']['best_model'])
    output_dir = Path(config['paths']['sagemaker_model'])
    output_dir.mkdir(parents=True, exist_ok=True)

    assert model_dir.exists(), f'Best model not found at {model_dir}'
    adapter_files = list(model_dir.iterdir())
    print(f'Packaging {len(adapter_files)} adapter files from {model_dir}')
    for f in adapter_files: print(f'  {f.name} ({f.stat().st_size/1024:.1f} KB)')

    staging_dir = output_dir/'staging'
    if staging_dir.exists(): shutil.rmtree(staging_dir)
    staging_dir.mkdir()
    code_dir = staging_dir/'code'
    code_dir.mkdir()

    for f in model_dir.iterdir():
        if f.is_file(): shutil.copy2(f, staging_dir/f.name)
        elif f.is_dir(): shutil.copytree(f, staging_dir/f.name)

    # SageMaker inference handler — 4 required functions: model_fn, input_fn, predict_fn, output_fn
    inference_script = '''
import base64, io, json, torch
from peft import PeftModel
from PIL import Image
from transformers import AutoModelForCausalLM, AutoProcessor

def model_fn(model_dir):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    processor = AutoProcessor.from_pretrained(model_dir, trust_remote_code=True)
    base = AutoModelForCausalLM.from_pretrained("microsoft/Florence-2-base-ft", trust_remote_code=True, torch_dtype=torch.float16, cache_dir="/tmp/hf_cache")
    model = PeftModel.from_pretrained(base, model_dir).to(device).eval()
    return {"model": model, "processor": processor, "device": device}

def input_fn(request_body, content_type):
    data = json.loads(request_body)
    return {"image_bytes": base64.b64decode(data["image"]), "prompt": data.get("prompt", "<CROP_DISEASE>")}

def predict_fn(input_data, model_dict):
    model, processor, device = model_dict["model"], model_dict["processor"], model_dict["device"]
    image = Image.open(io.BytesIO(input_data["image_bytes"])).convert("RGB")
    inputs = processor(text=input_data["prompt"], images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512, num_beams=3)
    text = processor.batch_decode(out, skip_special_tokens=True)[0]
    return {"generated_text": text, "disease_name": text.split(". ", 1)[0].strip(), "confidence": 0.85}

def output_fn(prediction, accept):
    return json.dumps(prediction), "application/json"
'''
    with open(code_dir/'inference.py','w') as f: f.write(inference_script)
    with open(code_dir/'requirements.txt','w') as f: f.write('peft>=0.12.0\ntransformers>=4.44.0\ntorch>=2.4.0\ntimm>=1.0.0\neinops>=0.8.0\nPillow>=10.0.0\n')
    print(f'  ✓ inference.py written ({len(inference_script)} chars)')
    print(f'  ✓ requirements.txt written')

    tar_path = output_dir/'model.tar.gz'
    with tarfile.open(tar_path,'w:gz') as tar:
        for item in staging_dir.iterdir(): tar.add(item, arcname=item.name)
    shutil.rmtree(staging_dir)

    size_mb = tar_path.stat().st_size/1024/1024
    print(f'\n✓ model.tar.gz created: {tar_path} ({size_mb:.1f} MB)')
    return tar_path

tar_path = package_model(config)

In [ ]:
# upload_to_s3() and deploy_endpoint()
def upload_to_s3(tar_path, config):
    s3_path = config['sagemaker']['model_s3_path']
    bucket = s3_path.replace('s3://','').split('/')[0]
    key = '/'.join(s3_path.replace('s3://','').split('/')[1:]) + 'model.tar.gz'
    print(f'Uploading {tar_path} ({tar_path.stat().st_size/1024/1024:.1f} MB) ...')
    boto3.client('s3').upload_file(str(tar_path), bucket, key)
    uri = f's3://{bucket}/{key}'
    print(f'✓ Uploaded to {uri}')
    return uri

def deploy_endpoint(s3_uri, config):
    sm = boto3.client('sagemaker')
    endpoint_name = config['sagemaker']['endpoint_name']
    model_name    = f'{endpoint_name}-model'
    config_name   = f'{endpoint_name}-config'
    print(f'Deploying endpoint: {endpoint_name}')

    role_arn = boto3.client('iam').get_role(RoleName='AmazonSageMaker-ExecutionRole-20260329T000445')['Role']['Arn']
    print(f'  IAM role: {role_arn}')

    try: sm.delete_model(ModelName=model_name); print(f'  Deleted old model: {model_name}')
    except: pass
    sm.create_model(ModelName=model_name,PrimaryContainer={'Image':'763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-inference:2.0.0-transformers4.28.1-gpu-py310-cu118-ubuntu20.04','ModelDataUrl':s3_uri,'Environment':{'SAGEMAKER_PROGRAM':'inference.py','SAGEMAKER_SUBMIT_DIRECTORY':s3_uri}},ExecutionRoleArn=role_arn)
    print(f'  ✓ SageMaker model created: {model_name}')

    try: sm.delete_endpoint_config(EndpointConfigName=config_name); print(f'  Deleted old config')
    except: pass
    sm_cfg = config['sagemaker']
    sm.create_endpoint_config(EndpointConfigName=config_name,ProductionVariants=[{'VariantName':'primary','ModelName':model_name,'ServerlessConfig':{'MemorySizeInMB':sm_cfg['serverless']['memory_size_mb'],'MaxConcurrency':sm_cfg['serverless']['max_concurrency']}}])
    print(f'  ✓ Endpoint config created: {config_name}')
    print(f'  Memory: {sm_cfg["serverless"]["memory_size_mb"]} MB | Max concurrency: {sm_cfg["serverless"]["max_concurrency"]}')

    try:
        sm.describe_endpoint(EndpointName=endpoint_name)
        sm.update_endpoint(EndpointName=endpoint_name, EndpointConfigName=config_name)
        print(f'  ✓ Updating existing endpoint: {endpoint_name}')
    except sm.exceptions.ClientError:
        sm.create_endpoint(EndpointName=endpoint_name, EndpointConfigName=config_name)
        print(f'  ✓ Creating new endpoint: {endpoint_name}')
    print(f'\n✓ Deployment initiated. Monitor at:')
    print(f'  https://console.aws.amazon.com/sagemaker/home#/endpoints/{endpoint_name}')

s3_uri = upload_to_s3(tar_path, config)
deploy_endpoint(s3_uri, config)

---
## 5. Image Preprocessor
**Production file:** `src/pipeline/image_preprocessor.py`

Validates and cleans images before sending to Florence-2. **EXIF stripping is a security requirement** — EXIF metadata can contain GPS coordinates and device info.

In [ ]:
ALLOWED_FORMATS = {'JPEG','PNG','WEBP'}
MAX_SIZE_BYTES = 5 * 1024 * 1024

def validate_image(image_bytes):
    """Returns (is_valid, message)."""
    if len(image_bytes) == 0: return False, 'Empty file'
    if len(image_bytes) > MAX_SIZE_BYTES: return False, f'File too large ({len(image_bytes)/1e6:.1f}MB). Max: 5MB'
    try:
        img = Image.open(io.BytesIO(image_bytes))
        if img.format not in ALLOWED_FORMATS: return False, f'Unsupported format: {img.format}'
        return True, 'valid'
    except Exception as e: return False, f'Invalid image: {e}'

def preprocess(image_bytes, target_size=224):
    """Resize + strip EXIF for privacy."""
    img = Image.open(io.BytesIO(image_bytes))
    # Strip EXIF by copying pixel data to new image
    data = list(img.getdata())
    clean = Image.new(img.mode, img.size)
    clean.putdata(data)
    clean = clean.convert('RGB').resize((target_size, target_size), Image.LANCZOS)
    buf = io.BytesIO()
    clean.save(buf, format='JPEG', quality=95)
    return buf.getvalue()

# Test with a real image
sample_path = list(Path('/tmp/plantvillage').rglob('*.JPG'))[0]
with open(sample_path,'rb') as f: raw_bytes = f.read()

valid, msg = validate_image(raw_bytes)
print(f'validate_image: {msg} (valid={valid})')
print(f'  Original size: {len(raw_bytes)/1024:.1f} KB')

processed = preprocess(raw_bytes, target_size=224)
print(f'  Processed size: {len(processed)/1024:.1f} KB')
img_check = Image.open(io.BytesIO(processed))
print(f'  Output shape: {img_check.size} (should be 224x224)')
assert img_check.size == (224,224), 'Resize failed!'
print('✓ validate_image and preprocess checks passed')

# Test edge cases
valid, msg = validate_image(b'')
assert not valid and 'Empty' in msg, 'Empty file check failed'
print('✓ Edge case: empty file correctly rejected')

valid, msg = validate_image(b'x' * (6*1024*1024))
assert not valid and 'large' in msg, 'Size check failed'
print('✓ Edge case: oversized file correctly rejected')

---
## 6. Florence-2 Inference Service
**Production file:** `src/models/florence_inference.py`

`FlorenceInferenceService` — invokes SageMaker endpoint in production.
`LocalFlorenceInference` — runs locally (no SageMaker needed), used for dev/testing.

In [ ]:
@dataclass
class DiagnosisResult:
    disease_name: str
    confidence: float
    confidence_level: str
    treatment_english: str
    raw_output: str
    inference_time_ms: float

class FlorenceInferenceService:
    """Invoke Florence-2 on SageMaker. Mirrors src/models/florence_inference.py."""
    def __init__(self):
        with open('configs/app_config.yaml') as f: self.config = yaml.safe_load(f)
        with open('configs/model_config.yaml') as f: self.model_config = yaml.safe_load(f)
        self.endpoint_name = self.config['model']['sagemaker_endpoint']
        self.thresholds = self.model_config['confidence_thresholds']
        self.client = boto3.client('sagemaker-runtime')
        print(f'  SageMaker endpoint: {self.endpoint_name}')
        print(f'  Confidence thresholds: {self.thresholds}')

    def _preprocess_image(self, image_bytes):
        return preprocess(image_bytes, self.config['image']['resize_to'])

    def _parse_confidence(self, raw_output):
        m = re.search(r'Confidence:\s*([\d.]+)', raw_output)
        return float(m.group(1)) if m else 0.75

    def _get_confidence_level(self, c):
        if c >= self.thresholds['high']:   return 'high'
        if c >= self.thresholds['medium']: return 'medium'
        if c >= self.thresholds['low']:    return 'low'
        return 'reject'

    def diagnose(self, image_bytes):
        start = time.monotonic()
        processed = self._preprocess_image(image_bytes)
        payload = {'image': base64.b64encode(processed).decode(), 'prompt': '<CROP_DISEASE>'}
        response = self.client.invoke_endpoint(EndpointName=self.endpoint_name, ContentType='application/json', Body=json.dumps(payload))
        result = json.loads(response['Body'].read())
        ms = (time.monotonic()-start)*1000
        raw = result.get('generated_text','')
        parts = raw.split('. ',1)
        conf = result.get('confidence', self._parse_confidence(raw))
        return DiagnosisResult(disease_name=parts[0].strip(),confidence=conf,confidence_level=self._get_confidence_level(conf),treatment_english=parts[1] if len(parts)>1 else '',raw_output=raw,inference_time_ms=ms)

class LocalFlorenceInference:
    """Local inference for dev/testing. Same logic as SageMaker endpoint, no AWS needed."""
    def __init__(self, model_path='outputs/florence2_lora/best'):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        assert Path(model_path).exists(), f'Model path not found: {model_path}'
        print(f'Loading local model from {model_path} on {self.device} ...')
        self.processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)
        base = AutoModelForCausalLM.from_pretrained('microsoft/Florence-2-base-ft', trust_remote_code=True, torch_dtype=torch.float16)
        self.model = PeftModel.from_pretrained(base, model_path).to(self.device).eval()
        print(f'✓ Local model loaded')

    def diagnose(self, image_bytes):
        start = time.monotonic()
        image = Image.open(io.BytesIO(image_bytes)).convert('RGB')
        inputs = self.processor(text='<CROP_DISEASE>', images=image, return_tensors='pt').to(self.device)
        with torch.no_grad():
            out = self.model.generate(**inputs, max_new_tokens=512, num_beams=3)
        raw = self.processor.batch_decode(out, skip_special_tokens=True)[0]
        ms = (time.monotonic()-start)*1000
        parts = raw.split('. ',1)
        return DiagnosisResult(disease_name=parts[0].strip() if parts else 'Unknown',confidence=0.85,confidence_level='medium',treatment_english=parts[1] if len(parts)>1 else '',raw_output=raw,inference_time_ms=ms)

print('✓ FlorenceInferenceService and LocalFlorenceInference defined')

In [ ]:
# Test local inference
import matplotlib.pyplot as plt
local_model = LocalFlorenceInference(model_path=str(best_model_dir))

with open(sample_path,'rb') as f: image_bytes = f.read()
result = local_model.diagnose(image_bytes)

print('=== Local Inference Result ===')
print(f'  Image:            {sample_path.name}')
print(f'  Disease:          {result.disease_name}')
print(f'  Confidence:       {result.confidence:.2f} ({result.confidence_level})')
print(f'  Treatment (100c): {result.treatment_english[:100]}...')
print(f'  Inference time:   {result.inference_time_ms:.0f}ms')
print(f'  Raw output (150c):{result.raw_output[:150]}')

assert result.disease_name != '', 'Disease name is empty!'
assert result.inference_time_ms > 0, 'Inference time should be > 0'
print('✓ Local inference checks passed')

plt.figure(figsize=(5,5))
plt.imshow(Image.open(sample_path))
plt.axis('off')
plt.title(f'{result.disease_name}\n({result.confidence:.0%} confidence)', fontsize=10)
plt.tight_layout(); plt.show()

---
## 7. S3 Service
**Production file:** `src/services/s3_service.py`

Every diagnosis request uploads the image to S3 — this creates a permanent audit trail and enables future retraining with real-world farmer images.

In [ ]:
class S3Service:
    """Mirrors src/services/s3_service.py."""
    def __init__(self):
        with open('configs/app_config.yaml') as f: config = yaml.safe_load(f)
        self.bucket = config['storage']['s3_bucket']
        self.prefix = config['storage']['s3_prefix']
        self.client = boto3.client('s3')
        print(f'  S3 bucket: {self.bucket}')
        print(f'  S3 prefix: {self.prefix}')

    def upload_image(self, image_bytes, request_id):
        """Upload image and return S3 key."""
        timestamp = datetime.utcnow().strftime('%Y/%m/%d')
        key = f'{self.prefix}{timestamp}/{request_id}.jpg'
        self.client.put_object(Bucket=self.bucket, Key=key, Body=image_bytes, ContentType='image/jpeg')
        print(f'  Uploaded: s3://{self.bucket}/{key}')
        return key

# Test
s3_service = S3Service()
test_id = str(uuid.uuid4())
key = s3_service.upload_image(image_bytes, test_id)
print(f'\n✓ S3 upload test passed')
print(f'  request_id: {test_id}')
print(f'  s3_key:     {key}')
assert test_id in key, 'request_id should be in S3 key'
print('✓ S3 key format check passed')

---
## 8. Feedback Service
**Production file:** `src/services/feedback_service.py`

Logs every prediction to DynamoDB. Farmers can submit feedback (correct/incorrect), which gets stored alongside the prediction for future model retraining.

In [ ]:
class FeedbackService:
    """Mirrors src/services/feedback_service.py."""
    def __init__(self):
        with open('configs/app_config.yaml') as f: config = yaml.safe_load(f)
        self.table_name = config['storage']['dynamodb_table']
        self.table = boto3.resource('dynamodb').Table(self.table_name)
        print(f'  DynamoDB table: {self.table_name}')

    def log_prediction(self, request_id, image_key, disease, confidence, treatment, language, inference_time_ms):
        try:
            self.table.put_item(Item={
                'PK': f'REQUEST#{request_id}', 'SK': 'PREDICTION',
                'disease': disease, 'confidence': str(confidence),
                'treatment_en': treatment[:1000], 'language': language,
                'image_s3_key': image_key, 'inference_time_ms': str(inference_time_ms),
                'timestamp': datetime.now(timezone.utc).isoformat(),
            })
            print(f'  ✓ Prediction logged: {request_id[:8]}... disease={disease}')
        except Exception as e:
            logger.error(f'Failed to log prediction: {e}')  # Non-blocking

    def submit_feedback(self, request_id, is_correct, actual_disease=None, comment=None):
        try:
            self.table.put_item(Item={
                'PK': f'REQUEST#{request_id}', 'SK': 'FEEDBACK',
                'is_correct': is_correct, 'actual_disease': actual_disease or '',
                'comment': comment or '', 'submitted_at': datetime.now(timezone.utc).isoformat(),
            })
            print(f'  ✓ Feedback recorded: {request_id[:8]}... correct={is_correct}')
        except Exception as e:
            logger.error(f'Failed to record feedback: {e}'); raise

# Test
feedback_service = FeedbackService()
feedback_service.log_prediction(test_id, key, result.disease_name, result.confidence, result.treatment_english, 'hi-IN', result.inference_time_ms)
feedback_service.submit_feedback(test_id, is_correct=True, comment='Great diagnosis!')
print('\n✓ Feedback service test passed')

---
## 9. Sarvam Translation Service
**Production file:** `src/services/sarvam_translate.py`

Uses Sarvam Mayura API. API key is fetched from **AWS Secrets Manager** — never hardcoded. Graceful degradation: if API fails, returns English.

In [ ]:
with open('configs/sarvam_config.yaml') as f: sarvam_config = yaml.safe_load(f)
print('=== Sarvam Config ===')
print(f'  Base URL:  {sarvam_config["api"]["base_url"]}')
print(f'  Timeout:   {sarvam_config["api"]["timeout"]}s')
print(f'  Retry delay: {sarvam_config["api"]["retry_delay_ms"]}ms')
print(f'  Translation model: {sarvam_config["translation"]["model"]}')
print(f'  TTS model:         {sarvam_config["tts"]["model"]}')
print(f'  TTS default speaker: {sarvam_config["tts"]["default_speaker"]}')
print(f'  Supported translation languages:')
for name,code in sarvam_config['translation']['supported_languages'].items():
    tts_ok = code in sarvam_config['tts']['supported_languages']
    print(f'    {name} ({code}) | TTS: {"✓" if tts_ok else "✗ falls back to Hindi"}')
print('✓ Sarvam config loaded')

In [ ]:
class SarvamTranslateService:
    """Mirrors src/services/sarvam_translate.py."""
    def __init__(self):
        self.config = sarvam_config
        self.base_url = self.config['api']['base_url']
        self.timeout  = self.config['api']['timeout']
        self.api_key  = self._load_api_key()
        self.headers  = {'API-Subscription-Key': self.api_key, 'Content-Type': 'application/json'}
        print(f'  API key loaded: {self.api_key[:6]}... (masked)')

    def _load_api_key(self):
        """Load from Secrets Manager, fall back to env var."""
        try:
            client = boto3.client('secretsmanager', region_name='us-east-1')
            secret = client.get_secret_value(SecretId='krishirakshak/api-keys')
            key = json.loads(secret['SecretString'])['SARVAM_API_KEY']
            print('  ✓ API key loaded from Secrets Manager')
            return key
        except Exception as e:
            logger.warning(f'Secrets Manager failed: {e}. Trying env var.')
            return os.environ.get('SARVAM_API_KEY', '')

    def translate(self, text, target_language):
        if target_language == 'en-IN':
            print(f'  Skipping translation (target is en-IN)')
            return {'translated_text': text, 'language': 'en-IN', 'latency_ms': 0}

        tr = self.config['translation']
        if target_language not in tr['supported_languages'].values():
            logger.warning(f'Unsupported: {target_language}, returning English')
            return {'translated_text': text, 'language': 'en-IN', 'latency_ms': 0}

        start = time.monotonic()
        try:
            payload = {'input':text,'source_language_code':tr['source_language'],'target_language_code':target_language,'model':tr['model'],'mode':tr['mode'],'enable_preprocessing':tr['enable_preprocessing']}
            with httpx.Client(timeout=self.timeout) as client:
                r = client.post(f'{self.base_url}{tr["endpoint"]}', headers=self.headers, json=payload)
                r.raise_for_status()
            ms = (time.monotonic()-start)*1000
            translated = r.json().get('translated_text', text)
            print(f'  ✓ Translated to {target_language}: {len(text)} → {len(translated)} chars ({ms:.0f}ms)')
            return {'translated_text': translated, 'language': target_language, 'latency_ms': ms}
        except Exception as e:
            logger.error(f'Translation failed: {e}. Returning English.')
            return {'translated_text': text, 'language': 'en-IN', 'latency_ms': 0, 'error': str(e)}

print('Initializing SarvamTranslateService...')
translator = SarvamTranslateService()

# Test translations
sample_text = result.treatment_english[:300]
print(f'\nInput text ({len(sample_text)} chars):')
print(f'  {sample_text[:100]}...')

for lang_code in ['hi-IN','mr-IN','ta-IN']:
    print(f'\nTranslating to {lang_code}...')
    t = translator.translate(sample_text, lang_code)
    print(f'  Result: {t["translated_text"][:100]}...')
    if 'error' not in t:
        assert len(t['translated_text']) > 0, 'Empty translation!'
    time.sleep(0.2)  # Sarvam free tier rate limit

# Test graceful degradation
t = translator.translate(sample_text, 'en-IN')
assert t['translated_text'] == sample_text, 'en-IN should return original text'
print('\n✓ en-IN passthrough check passed')

t = translator.translate(sample_text, 'xx-XX')
assert t['language'] == 'en-IN', 'Unsupported language should fall back to English'
print('✓ Unsupported language fallback check passed')

---
## 10. Sarvam TTS Service
**Production file:** `src/services/sarvam_tts.py`

Converts translated text to speech. Why this matters: millions of Indian farmers are not literate — audio output in their native language makes the tool accessible.

In [ ]:
from IPython.display import Audio, display

class SarvamTTSService:
    """Mirrors src/services/sarvam_tts.py."""
    def __init__(self):
        self.config     = sarvam_config
        self.base_url   = self.config['api']['base_url']
        self.timeout    = self.config['api']['timeout']
        self.tts_config = self.config['tts']
        self.api_key    = self._load_api_key()
        self.headers    = {'API-Subscription-Key': self.api_key, 'Content-Type': 'application/json'}
        print(f'  TTS model: {self.tts_config["model"]}')
        print(f'  Speaker: {self.tts_config["default_speaker"]}')
        print(f'  Sample rate: {self.tts_config["speech_sample_rate"]} Hz')

    def _load_api_key(self):
        try:
            client = boto3.client('secretsmanager', region_name='us-east-1')
            secret = client.get_secret_value(SecretId='krishirakshak/api-keys')
            return json.loads(secret['SecretString'])['SARVAM_API_KEY']
        except Exception:
            return os.environ.get('SARVAM_API_KEY', '')

    def synthesize(self, text, language, speaker=None):
        supported = self.tts_config['supported_languages']
        if language not in supported:
            fallback = self.tts_config.get('fallback_for_unsupported','hi-IN')
            print(f'  TTS not natively supported for {language}, falling back to {fallback}')
            language = fallback
        speaker = speaker or self.tts_config['default_speaker']
        start = time.monotonic()
        try:
            payload = {'inputs': text[:5000], 'target_language_code': language, 'speaker': speaker,
                      'model': self.tts_config['model'], 'pitch': self.tts_config['pitch'],
                      'pace': self.tts_config['pace'], 'loudness': self.tts_config['loudness'],
                      'speech_sample_rate': self.tts_config['speech_sample_rate'],
                      'enable_preprocessing': self.tts_config['enable_preprocessing']}
            with httpx.Client(timeout=self.timeout+5) as client:
                r = client.post(f'{self.base_url}{self.tts_config["endpoint"]}', headers=self.headers, json=payload)
                r.raise_for_status()
            ms = (time.monotonic()-start)*1000
            audios = r.json().get('audios',[])
            audio_b64 = audios[0] if audios else None
            print(f'  ✓ Audio generated for {language}, speaker={speaker} ({len(text)} chars, {ms:.0f}ms)')
            return {'audio_base64': audio_b64, 'format': 'wav', 'sample_rate': self.tts_config['speech_sample_rate'], 'language': language, 'speaker': speaker, 'latency_ms': ms}
        except Exception as e:
            logger.error(f'TTS failed: {e}')
            return {'audio_base64': None, 'latency_ms': 0, 'error': str(e)}

print('Initializing SarvamTTSService...')
tts_service = SarvamTTSService()

# Test TTS — Hindi
hindi = translator.translate(result.treatment_english[:200], 'hi-IN')
print(f'\nGenerating Hindi audio...')
audio_result = tts_service.synthesize(hindi['translated_text'], 'hi-IN')
if audio_result.get('audio_base64'):
    audio_bytes = base64.b64decode(audio_result['audio_base64'])
    print(f'  Audio size: {len(audio_bytes)/1024:.1f} KB')
    print(f'  Sample rate: {audio_result["sample_rate"]} Hz')
    display(Audio(data=audio_bytes, rate=audio_result['sample_rate']))
else:
    print(f'  TTS error: {audio_result.get("error")}')

# Test Marathi fallback to Hindi
print('\nTesting Marathi → Hindi TTS fallback...')
mr_audio = tts_service.synthesize('test', 'mr-IN')
assert mr_audio.get('language') == 'hi-IN', 'Marathi should fall back to Hindi'
print('✓ Marathi TTS fallback check passed')

---
## 11. Full End-to-End Pipeline
**Production file:** `src/pipeline/diagnosis_pipeline.py`

Orchestrates all steps with **graceful degradation** — if any step fails, it degrades rather than crashing:
- S3 upload fails → log warning, continue
- Translation fails → return English
- TTS fails → return text only, no audio
- Confidence < threshold → return 'uncertain'

In [ ]:
class DiagnosisPipeline:
    """Mirrors src/pipeline/diagnosis_pipeline.py."""
    def __init__(self, use_local_model=True):
        with open('configs/app_config.yaml') as f: self.config = yaml.safe_load(f)
        self.florence   = LocalFlorenceInference(str(best_model_dir)) if use_local_model else FlorenceInferenceService()
        self.translator = SarvamTranslateService()
        self.tts        = SarvamTTSService()
        self.s3         = S3Service()
        self.feedback   = FeedbackService()
        self.confidence_threshold = self.config['model']['confidence_threshold']
        self.use_bedrock = self.config['model']['bedrock_fallback']
        print(f'  Confidence threshold: {self.confidence_threshold}')
        print(f'  Bedrock fallback: {self.use_bedrock}')

    def _should_use_bedrock(self, diagnosis):
        if diagnosis.confidence < 0.6:                      return True
        if len(diagnosis.treatment_english) < 50:           return True
        if 'unknown' in diagnosis.disease_name.lower():     return True
        return False

    def _augment_with_bedrock(self, diagnosis):
        """Use Claude via Bedrock for India-specific treatment advice."""
        try:
            bedrock = boto3.client('bedrock-runtime', region_name=self.config['model']['bedrock_region'])
            prompt = (f'You are an agricultural expert advising Indian farmers. '
                      f'Crop diagnosed with: {diagnosis.disease_name}. '
                      f'Provide: 1) Immediate action 2) Pesticide (Indian brand names) 3) Dosage 4) Prevention tips')
            response = bedrock.invoke_model(modelId=self.config['model']['bedrock_model_id'],body=json.dumps({'anthropic_version':'bedrock-2023-05-31','max_tokens':self.config['model']['bedrock_max_tokens'],'messages':[{'role':'user','content':prompt}]}))
            advice = json.loads(response['body'].read())['content'][0]['text']
            print(f'  ✓ Bedrock augmentation: {len(advice)} chars')
            return advice
        except Exception as e:
            logger.warning(f'Bedrock failed: {e}. Using Florence-2 output.')
            return diagnosis.treatment_english

    def run(self, image_bytes, language='en-IN', include_audio=True):
        request_id = str(uuid.uuid4())
        start = time.monotonic()
        print(f'\n[Pipeline] request_id={request_id[:8]}... language={language}')

        # Step 1: Upload to S3
        image_key = ''
        try:
            print('[Step 1] Uploading image to S3...')
            image_key = self.s3.upload_image(image_bytes, request_id)
        except Exception as e:
            logger.warning(f'S3 upload failed (non-critical): {e}')

        # Step 2: Florence-2 inference
        print('[Step 2] Running Florence-2 inference...')
        try:
            diagnosis = self.florence.diagnose(image_bytes)
            print(f'  Disease: {diagnosis.disease_name} | Confidence: {diagnosis.confidence:.2f} ({diagnosis.confidence_level}) | {diagnosis.inference_time_ms:.0f}ms')
        except Exception as e:
            print(f'  ✗ Inference failed: {e}')
            return {'request_id':request_id,'error':'MODEL_UNAVAILABLE','message':str(e)}

        # Step 3: Confidence check
        print(f'[Step 3] Confidence check ({diagnosis.confidence:.2f} vs threshold {self.confidence_threshold})')
        if diagnosis.confidence < self.confidence_threshold:
            print(f'  ✗ Below threshold — returning uncertain response')
            return {'request_id':request_id,'disease':{'name':'uncertain','confidence':diagnosis.confidence,'confidence_level':'low'},'treatment':{'english':'Confidence too low. Please consult an agricultural expert.'},'audio':None}
        print(f'  ✓ Confidence OK')

        # Step 4: Bedrock augmentation (optional)
        treatment_en = diagnosis.treatment_english
        print(f'[Step 4] Bedrock augmentation: {self.use_bedrock and self._should_use_bedrock(diagnosis)}')
        if self.use_bedrock and self._should_use_bedrock(diagnosis):
            treatment_en = self._augment_with_bedrock(diagnosis)

        # Step 5: Translation
        print(f'[Step 5] Translating to {language}...')
        translation = {'translated_text': treatment_en, 'language': 'en-IN', 'latency_ms': 0}
        if language != 'en-IN':
            try: translation = self.translator.translate(treatment_en, language)
            except Exception as e: logger.warning(f'Translation failed, using English: {e}')

        # Step 6: TTS
        print(f'[Step 6] Generating TTS audio (include_audio={include_audio})...')
        audio_result = None
        if include_audio:
            try:
                audio_result = self.tts.synthesize(translation['translated_text'], language)
                if audio_result.get('error'): audio_result = None
            except Exception as e: logger.warning(f'TTS failed: {e}')

        # Step 7: Log to DynamoDB
        print(f'[Step 7] Logging prediction to DynamoDB...')
        try:
            self.feedback.log_prediction(request_id,image_key,diagnosis.disease_name,diagnosis.confidence,treatment_en,language,diagnosis.inference_time_ms)
        except Exception as e: logger.warning(f'Logging failed (non-critical): {e}')

        total_ms = (time.monotonic()-start)*1000
        print(f'\n[Pipeline DONE] Total time: {total_ms:.0f}ms')

        return {
            'request_id': request_id,
            'disease': {'name':diagnosis.disease_name,'confidence':round(diagnosis.confidence,3),'confidence_level':diagnosis.confidence_level},
            'treatment': {'english':treatment_en,'translated':translation['translated_text'],'language':translation['language']},
            'audio': {'base64':audio_result['audio_base64'],'format':audio_result.get('format','wav'),'sample_rate':audio_result.get('sample_rate',22050)} if audio_result and audio_result.get('audio_base64') else None,
            'metadata': {'request_id':request_id,'inference_ms':round(diagnosis.inference_time_ms,1),'total_ms':round(total_ms,1)},
        }

print('✓ DiagnosisPipeline defined')

In [ ]:
# Run full pipeline
print('Initializing pipeline...')
pipeline = DiagnosisPipeline(use_local_model=True)

with open(sample_path,'rb') as f: image_bytes = f.read()
final = pipeline.run(image_bytes, language='hi-IN', include_audio=True)

print('\n' + '='*60)
print('FINAL PIPELINE RESULT')
print('='*60)
print(f'Request ID:         {final["request_id"]}')
print(f'Disease:            {final["disease"]["name"]}')
print(f'Confidence:         {final["disease"]["confidence"]} ({final["disease"]["confidence_level"]})')
print(f'Treatment (English):{final["treatment"]["english"][:120]}...')
print(f'Treatment (Hindi):  {final["treatment"]["translated"][:120]}...')
print(f'Audio:              {"Generated" if final["audio"] else "Not available"}')
print(f'Inference time:     {final["metadata"]["inference_ms"]}ms')
print(f'Total time:         {final["metadata"]["total_ms"]}ms')

assert 'disease' in final, 'disease key missing from response'
assert 'treatment' in final, 'treatment key missing'
assert 'metadata' in final, 'metadata key missing'
assert final['metadata']['total_ms'] > 0
print('\n✓ All response structure checks passed')

if final.get('audio') and final['audio']['base64']:
    audio_bytes = base64.b64decode(final['audio']['base64'])
    print(f'\nPlaying Hindi audio ({len(audio_bytes)/1024:.1f} KB):')
    display(Audio(data=audio_bytes, rate=final['audio']['sample_rate']))

---
## 12. API Layer
**Production files:** `src/api/schemas.py`, `src/api/routes.py`, `src/api/middleware.py`, `src/api/main.py`

The FastAPI layer exposes the pipeline as a REST API. Three endpoints:
- `POST /v1/diagnose` — main diagnosis endpoint
- `POST /v1/feedback` — farmer feedback submission
- `GET /v1/health` — health check for load balancer

In [ ]:
from pydantic import BaseModel, Field

# --- schemas.py — Pydantic models define the API contract ---
class DiseaseInfo(BaseModel):
    name: str
    crop: str = ''
    severity: str = ''
    confidence: float = Field(ge=0, le=1)
    confidence_level: str

class TreatmentInfo(BaseModel):
    english: str
    translated: str = ''
    language: str = 'en-IN'

class AudioInfo(BaseModel):
    base64: str
    format: str = 'wav'
    sample_rate: int = 22050

class MetadataInfo(BaseModel):
    request_id: str
    model_version: str
    inference_time_ms: float
    total_time_ms: float
    timestamp: str

class DiagnoseResponse(BaseModel):
    request_id: str
    disease: DiseaseInfo
    treatment: TreatmentInfo
    audio: AudioInfo | None = None
    metadata: MetadataInfo

class FeedbackRequest(BaseModel):
    request_id: str
    is_correct: bool
    actual_disease: str | None = None
    comment: str | None = None

class FeedbackResponse(BaseModel):
    status: str = 'recorded'
    request_id: str
    message: str = 'Thank you for your feedback. This helps improve our system.'

class HealthResponse(BaseModel):
    status: str
    model_version: str
    sagemaker_endpoint: str
    sarvam_api: str
    uptime_seconds: int
    timestamp: str

class LanguageInfo(BaseModel):
    code: str
    name: str
    tts_supported: bool

class LanguagesResponse(BaseModel):
    languages: list[LanguageInfo]

# Test schema validation
test_feedback = FeedbackRequest(request_id='abc-123', is_correct=True, comment='Works great')
print(f'✓ FeedbackRequest: {test_feedback.model_dump()}')

try:
    DiseaseInfo(name='test', confidence=1.5, confidence_level='high')  # should fail
    print('✗ Should have rejected confidence > 1')
except Exception as e:
    print(f'✓ Schema validation working: confidence > 1 correctly rejected')

print('✓ All Pydantic schemas defined and validated')

In [ ]:
# middleware.py — adds request ID and timing headers to every response
from starlette.middleware.base import BaseHTTPMiddleware
from starlette.requests import Request as StarletteRequest
from starlette.responses import Response as StarletteResponse

class RequestIDMiddleware(BaseHTTPMiddleware):
    """Mirrors src/api/middleware.py."""
    async def dispatch(self, request: StarletteRequest, call_next):
        request_id = str(uuid.uuid4())
        request.state.request_id = request_id
        start = time.monotonic()
        response: StarletteResponse = await call_next(request)
        duration_ms = (time.monotonic()-start)*1000
        response.headers['X-Request-ID'] = request_id
        response.headers['X-Inference-Time-Ms'] = str(int(duration_ms))
        logger.info(f'{request.method} {request.url.path} status={response.status_code} duration={duration_ms:.0f}ms request_id={request_id}')
        return response

print('✓ RequestIDMiddleware defined')

In [ ]:
# main.py — FastAPI app wiring
import structlog
from fastapi import FastAPI, File, Form, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware

structlog.configure(processors=[structlog.processors.TimeStamper(fmt='iso'), structlog.processors.JSONRenderer()])

with open('configs/app_config.yaml') as f: app_config = yaml.safe_load(f)

app = FastAPI(
    title=app_config['app']['name'],
    version=app_config['app']['version'],
    description=app_config['app']['description'],
    docs_url='/docs', redoc_url='/redoc',
)
app.add_middleware(CORSMiddleware, allow_origins=app_config['server']['cors_origins'], allow_credentials=True, allow_methods=['*'], allow_headers=['*'])
app.add_middleware(RequestIDMiddleware)

_pipeline_instance = None
_api_start_time = time.monotonic()

def get_pipeline():
    global _pipeline_instance
    if _pipeline_instance is None:
        _pipeline_instance = DiagnosisPipeline(use_local_model=True)
    return _pipeline_instance

@app.get('/')
async def root():
    return {'service': app_config['app']['name'], 'version': app_config['app']['version'], 'docs': '/docs'}

@app.post('/v1/diagnose')
async def diagnose(image: UploadFile = File(...), language: str = Form('en-IN'), include_audio: bool = Form(True)):
    if image.content_type not in ('image/jpeg','image/png','image/webp'):
        raise HTTPException(400, {'error':'UNSUPPORTED_FORMAT','message':f'Use JPEG or PNG. Got: {image.content_type}'})
    image_bytes = await image.read()
    if len(image_bytes) > 5*1024*1024:
        raise HTTPException(400, {'error':'INVALID_IMAGE','message':f'Too large ({len(image_bytes)/1e6:.1f}MB). Max 5MB.'})
    if len(image_bytes) == 0:
        raise HTTPException(400, {'error':'INVALID_IMAGE','message':'Empty file.'})
    result = get_pipeline().run(image_bytes, language=language, include_audio=include_audio)
    if 'error' in result: raise HTTPException(503, result)
    return result

@app.post('/v1/feedback')
async def submit_feedback(request: FeedbackRequest):
    feedback_svc = FeedbackService()
    feedback_svc.submit_feedback(request.request_id, request.is_correct, request.actual_disease, request.comment)
    return FeedbackResponse(request_id=request.request_id)

@app.get('/v1/health')
async def health_check():
    return HealthResponse(status='healthy', model_version='1.1.0', sagemaker_endpoint='active', sarvam_api='reachable', uptime_seconds=int(time.monotonic()-_api_start_time), timestamp=datetime.now(timezone.utc).isoformat())

@app.get('/v1/languages')
async def list_languages():
    tts_supported = set(sarvam_config['tts']['supported_languages'])
    languages = [LanguageInfo(code='en-IN', name='English', tts_supported=True)]
    for name,code in sarvam_config['translation']['supported_languages'].items():
        languages.append(LanguageInfo(code=code, name=name, tts_supported=code in tts_supported))
    return LanguagesResponse(languages=languages)

print(f'✓ FastAPI app created: {app_config["app"]["name"]} v{app_config["app"]["version"]}')
print(f'  Routes registered:')
for route in app.routes:
    if hasattr(route,'methods'): print(f'    {list(route.methods)} {route.path}')
print('  Run locally with: uvicorn src.api.main:app --reload')

---
## 13. Monitoring
**Production file:** `src/monitoring/metrics.py`

Publishes custom metrics to CloudWatch. These power dashboards and alerts:
- **InferenceLatencyMs** — alert if p99 > 10s
- **PredictionConfidence** — alert if avg < 0.4
- **FeedbackPositive/Negative** — track model quality over time
- **RequestCount** — business metric (requests per language per day)

In [ ]:
with open('configs/monitoring_config.yaml') as f: monitoring_config = yaml.safe_load(f)
print('=== Monitoring Config ===')
print(f'  CloudWatch namespace: {monitoring_config["cloudwatch"]["namespace"]}')
print(f'  Region: {monitoring_config["cloudwatch"]["region"]}')
print(f'  Metrics tracked:')
for metric in ['InferenceLatencyMs','PredictionConfidence','FeedbackPositive','FeedbackNegative','RequestCount']:
    print(f'    {metric}')
print('✓ Monitoring config loaded')

In [ ]:
class MetricsPublisher:
    """Mirrors src/monitoring/metrics.py."""
    def __init__(self):
        self.namespace = monitoring_config['cloudwatch']['namespace']
        self.client = boto3.client('cloudwatch', region_name=monitoring_config['cloudwatch']['region'])
        print(f'  Namespace: {self.namespace}')

    def put_metric(self, name, value, unit='None', dimensions=None):
        try:
            dim_list = [{'Name':k,'Value':str(v)} for k,v in (dimensions or {}).items()]
            self.client.put_metric_data(Namespace=self.namespace, MetricData=[{'MetricName':name,'Value':value,'Unit':unit,'Dimensions':dim_list}])
            print(f'  ✓ Published {name}={value} {unit} dims={dimensions}')
        except Exception as e:
            logger.warning(f'Failed to publish metric {name}: {e}')

    def record_inference(self, latency_ms, confidence, disease_class, model_version='1.1.0'):
        print(f'Recording inference metrics: latency={latency_ms:.0f}ms, confidence={confidence:.2f}, class={disease_class}')
        self.put_metric('InferenceLatencyMs', latency_ms, 'Milliseconds', {'ModelVersion':model_version,'DiseaseClass':disease_class})
        self.put_metric('PredictionConfidence', confidence, 'None', {'DiseaseClass':disease_class})

    def record_translation(self, latency_ms, language):
        print(f'Recording translation metric: latency={latency_ms:.0f}ms, lang={language}')
        self.put_metric('TranslationLatencyMs', latency_ms, 'Milliseconds', {'TargetLanguage':language})

    def record_feedback(self, is_positive):
        name = 'FeedbackPositive' if is_positive else 'FeedbackNegative'
        print(f'Recording feedback: {name}')
        self.put_metric(name, 1, 'Count')

    def record_request(self, endpoint, language):
        print(f'Recording request: endpoint={endpoint}, lang={language}')
        self.put_metric('RequestCount', 1, 'Count', {'Endpoint':endpoint,'Language':language})

# Test metrics
print('Initializing MetricsPublisher...')
metrics = MetricsPublisher()

print('\nPublishing test metrics to CloudWatch...')
metrics.record_inference(latency_ms=final['metadata']['inference_ms'], confidence=final['disease']['confidence'], disease_class=final['disease']['name'])
metrics.record_request(endpoint='/v1/diagnose', language='hi-IN')
metrics.record_feedback(is_positive=True)
print('\n✓ All metrics published to CloudWatch')
print('  View at: https://console.aws.amazon.com/cloudwatch/home#metricsV2:namespace=KrishiRakshak')

---
## Summary

```
S3 (raw images ~162K)
        │
        ▼
prepare_dataset.py  →  JSONL: {image_path, prefix, suffix} × 162K
        │
        ▼
fine_tune_florence.py  →  Florence-2 + LoRA, 10 epochs, val_loss checkpoint
        │
        ▼
evaluate_model.py  →  Accuracy/F1 on test set, 90% pass/fail gate
        │
        ▼
push_to_sagemaker.py  →  model.tar.gz → S3 → Serverless Endpoint
        │
        ▼
image_preprocessor.py  →  validate + resize 224×224 + strip EXIF
        │
        ▼
florence_inference.py  →  SageMaker invoke → DiagnosisResult
        │
        ▼
sarvam_translate.py  →  English → Hindi/Marathi/Tamil/...
        │
        ▼
sarvam_tts.py  →  Text → WAV audio (Bulbul v1)
        │
        ▼
diagnosis_pipeline.py  →  Orchestrates all steps with graceful degradation
        │
        ▼
src/api/  →  FastAPI: POST /v1/diagnose, POST /v1/feedback, GET /v1/health
        │
        ▼
s3_service.py + feedback_service.py  →  S3 (images) + DynamoDB (logs)
        │
        ▼
monitoring/metrics.py  →  CloudWatch: latency, confidence, feedback, requests
```